In [3]:
import nflreadpy as nfl
import pandas as pd

pd.set_option("display.max_columns", None)

import os
import django

os.environ.setdefault(
    "DJANGO_SETTINGS_MODULE", "backend.doritostats.settings"
)  # Adjust if your settings module is different
django.setup()

import os

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# Load environment variables
import dotenv

dotenv.load_dotenv(override=True)

from django.core.cache import cache

cache.clear()

# Query postgres

In [2]:
import os

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"


from backend.playoff_pool.models import CachedPlayerStats, CachedPlayers

# # Clear cache
# deleted_stats = CachedPlayerStats.objects.all().delete()[0]
# deleted_players = CachedPlayers.objects.all().delete()[0]

# Get all players from the cache
players = CachedPlayerStats.objects.filter(league=20)
players

<QuerySet [<CachedPlayerStats: Matthew Stafford - League 20 - Week 19>, <CachedPlayerStats: Brandon McManus - League 20 - Week 19>, <CachedPlayerStats: Chris Boswell - League 20 - Week 19>, <CachedPlayerStats: Brandin Cooks - League 20 - Week 19>, <CachedPlayerStats: Davante Adams - League 20 - Week 19>, <CachedPlayerStats: Stefon Diggs - League 20 - Week 19>, <CachedPlayerStats: Ka'imi Fairbairn - League 20 - Week 19>, <CachedPlayerStats: Hunter Henry - League 20 - Week 19>, <CachedPlayerStats: Tyler Higbee - League 20 - Week 19>, <CachedPlayerStats: Christian McCaffrey - League 20 - Week 19>, <CachedPlayerStats: George Kittle - League 20 - Week 19>, <CachedPlayerStats: Jake Elliott - League 20 - Week 19>, <CachedPlayerStats: Dallas Goedert - League 20 - Week 19>, <CachedPlayerStats: Dalton Schultz - League 20 - Week 19>, <CachedPlayerStats: DJ Moore - League 20 - Week 19>, <CachedPlayerStats: Saquon Barkley - League 20 - Week 19>, <CachedPlayerStats: Josh Allen - League 20 - Week 19>

In [ ]:
CachedPlayerStats.objects.filter(league=20).first().most_recent_game_id

<CachedPlayerStats: Matthew Stafford - League 20 - Week 19>

In [6]:
CachedPlayerStats.objects.filter(league=20).first()

from backend.playoff_pool.scoring import get_most_recent_game

get_most_recent_game(2025)

game_id     2025_20_LA_CHI
gameday         2026-01-18
gametime             18:30
Name: 280, dtype: object

In [5]:
from backend.playoff_pool.models import League
from backend.playoff_pool.cache_utils import refresh_league_cache

league = League.objects.get(id=21)
refresh_league_cache(league)

Starting cache refresh for league 21 (MP Fantasy Football)
Cache refresh complete for league 21 (MP Fantasy Football): 80 players cached, 145 game stats created, most recent game: 2025_21_LA_SEA


In [1]:
# What is the RAM usage of Numpy
import os, psutil

process = psutil.Process(os.getpid())
print(f"RAM usage of the process: {process.memory_info().rss / (1024 * 1024):.2f} MB")

import numpy as np
print(f"RAM usage of Numpy: {process.memory_info().rss / (1024 * 1024):.2f} MB")


import nflreadpy as nfl
print(f"RAM usage of nflreadpy: {process.memory_info().rss / (1024 * 1024):.2f} MB")

RAM usage of the process: 66.44 MB


RAM usage of Numpy: 73.75 MB
RAM usage of nflreadpy: 107.31 MB


In [ ]:
import time

start_time = time.time()
import nflreadpy as nfl

end_time = time.time()

print(f"Time taken to import nflreadpy: {end_time - start_time:.2f} seconds")

Time taken to import nflreadpy: 0.00 seconds


# Defense stats

In [32]:
from backend.playoff_pool.scoring import DEFENSE_SCORING_MULTIPLIERS

year = 2025
# Load all available team level stats
defense_stats = nfl.load_team_stats(seasons=[year]).to_pandas()

defense_stats["gsis_id"] = defense_stats["team"]
defense_stats["full_name"] = defense_stats["team"] + " D/ST"
defense_stats["position"] = "DST"
defense_stats["player_id"] = defense_stats["gsis_id"]

defense_stats = defense_stats[
    [
        "season",
        "week",
        "gsis_id",
        "full_name",
        "position",
        "player_id",
        "team",
        "season_type",
    ]
    + [
        key
        for key in DEFENSE_SCORING_MULTIPLIERS.keys()
        if key in defense_stats.columns
    ]
]
defense_stats.tail(10)

schedule = nfl.load_schedules(seasons=[year]).to_pandas()

# Create mapping for home teams (opponent score = away_score)
home_mapping = (
    schedule[["season", "week", "home_team", "away_score"]]
    .copy()
    .rename(columns={"home_team": "team", "away_score": "points_allowed"})
)

# Create mapping for away teams (opponent score = home_score)
away_mapping = (
    schedule[["season", "week", "away_team", "home_score"]]
    .copy()
    .rename(columns={"away_team": "team", "home_score": "points_allowed"})
)

# Combine both mappings
score_mapping = pd.concat([home_mapping, away_mapping], ignore_index=True)

# Merge with team stats
defense_stats = defense_stats.merge(
    score_mapping, on=["season", "week", "team"], how="left"
)
defense_stats

,season,week,gsis_id,full_name,position,player_id,team,season_type,special_teams_tds,def_tds,fumble_recovery_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_fumbles,def_interceptions,def_interception_yards,def_sacks,def_sack_yards,def_qb_hits,def_pass_defended,def_safeties,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,misc_yards,penalties,penalty_yards,timeouts,points_allowed
0,2025,1,ARI,ARI D/ST,DST,ARI,ARI,REG,0,0,0,36,2,36,3,10,0,0,0,0,1.0,6.0,2,10,0,0,0,0,0,3,30,3,73,0,9,54,0,13.0
1,2025,1,ATL,ATL D/ST,DST,ATL,ATL,REG,0,0,0,27,1,25,2,3,0,0,0,0,1.0,8.0,3,7,0,1,2,0,0,2,29,5,125,0,8,55,6,23.0
2,2025,1,BAL,BAL D/ST,DST,BAL,BAL,REG,0,0,0,51,2,22,3,8,1,0,0,0,1.0,5.0,3,4,0,1,1,0,0,1,6,6,167,0,7,51,5,41.0
3,2025,1,BUF,BUF D/ST,DST,BUF,BUF,REG,0,0,0,44,0,8,5,45,2,0,0,0,2.0,15.0,4,1,0,0,0,1,1,1,0,7,193,0,5,38,6,40.0
4,2025,1,CAR,CAR D/ST,DST,CAR,CAR,REG,0,0,0,42,0,20,1,2,0,0,1,10,0.0,0.0,1,4,0,0,0,0,0,2,0,2,43,0,4,35,6,26.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551,2025,19,LAC,LAC D/ST,DST,LAC,LAC,POST,0,0,0,38,0,24,7,36,2,0,1,2,5.0,33.0,5,2,0,1,1,1,1,0,0,5,128,0,4,20,4,16.0
552,2025,19,NE,NE D/ST,DST,NE,NE,POST,0,0,0,36,0,34,6,40,2,0,0,0,6.0,39.0,11,5,0,1,1,1,1,2,2,2,48,0,2,35,5,3.0
553,2025,19,PHI,PHI D/ST,DST,PHI,PHI,POST,0,0,0,34,1,21,3,7,1,0,2,3,1.0,5.0,2,5,0,0,0,0,0,0,0,2,61,0,7,48,5,23.0
554,2025,19,PIT,PIT D/ST,DST,PIT,PIT,POST,0,0,0,35,0,44,3,7,2,0,1,0,3.0,6.0,3,6,0,1,1,2,2,1,14,4,100,0,3,24,2,30.0


In [ ]:
from backend.playoff_pool.scoring import DEFENSE_SCORING_MULTIPLIERS

points_allowed_ranges = [
    key for key in DEFENSE_SCORING_MULTIPLIERS.keys() if "points_allowed_" in key
]

for pa_range in points_allowed_ranges:
    min_max = pa_range.replace("points_allowed_", "").split("_")
    print(min_max)
    if min_max[0] == "0":
        defense_stats[pa_range] = (defense_stats["points_allowed"] == 0).astype(int)
    elif min_max[1] == "plus":
        defense_stats[pa_range] = (
            defense_stats["points_allowed"] >= int(min_max[0])
        ).astype(int)
    else:
        defense_stats[pa_range] = (
            (defense_stats["points_allowed"] >= int(min_max[0]))
            & (defense_stats["points_allowed"] <= int(min_max[1]))
        ).astype(int)

defense_stats

['0']
['1', '6']
['7', '13']
['14', '20']
['21', '27']
['28', '34']
['35', 'plus']


,season,week,gsis_id,full_name,position,player_id,team,season_type,special_teams_tds,def_tds,fumble_recovery_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_fumbles,def_interceptions,def_interception_yards,def_sacks,def_sack_yards,def_qb_hits,def_pass_defended,def_safeties,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,misc_yards,penalties,penalty_yards,timeouts,points_allowed,points_allowed_0,points_allowed_1_6,points_allowed_7_13,points_allowed_14_20,points_allowed_21_27,points_allowed_28_34,points_allowed_35_plus
0,2025,1,ARI,ARI D/ST,DST,ARI,ARI,REG,0,0,0,36,2,36,3,10,0,0,0,0,1.0,6.0,2,10,0,0,0,0,0,3,30,3,73,0,9,54,0,13.0,0,0,1,0,0,0,0
1,2025,1,ATL,ATL D/ST,DST,ATL,ATL,REG,0,0,0,27,1,25,2,3,0,0,0,0,1.0,8.0,3,7,0,1,2,0,0,2,29,5,125,0,8,55,6,23.0,0,0,0,0,1,0,0
2,2025,1,BAL,BAL D/ST,DST,BAL,BAL,REG,0,0,0,51,2,22,3,8,1,0,0,0,1.0,5.0,3,4,0,1,1,0,0,1,6,6,167,0,7,51,5,41.0,0,0,0,0,0,0,1
3,2025,1,BUF,BUF D/ST,DST,BUF,BUF,REG,0,0,0,44,0,8,5,45,2,0,0,0,2.0,15.0,4,1,0,0,0,1,1,1,0,7,193,0,5,38,6,40.0,0,0,0,0,0,0,1
4,2025,1,CAR,CAR D/ST,DST,CAR,CAR,REG,0,0,0,42,0,20,1,2,0,0,1,10,0.0,0.0,1,4,0,0,0,0,0,2,0,2,43,0,4,35,6,26.0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551,2025,19,LAC,LAC D/ST,DST,LAC,LAC,POST,0,0,0,38,0,24,7,36,2,0,1,2,5.0,33.0,5,2,0,1,1,1,1,0,0,5,128,0,4,20,4,16.0,0,0,0,1,0,0,0
552,2025,19,NE,NE D/ST,DST,NE,NE,POST,0,0,0,36,0,34,6,40,2,0,0,0,6.0,39.0,11,5,0,1,1,1,1,2,2,2,48,0,2,35,5,3.0,0,1,0,0,0,0,0
553,2025,19,PHI,PHI D/ST,DST,PHI,PHI,POST,0,0,0,34,1,21,3,7,1,0,2,3,1.0,5.0,2,5,0,0,0,0,0,0,0,2,61,0,7,48,5,23.0,0,0,0,0,1,0,0
554,2025,19,PIT,PIT D/ST,DST,PIT,PIT,POST,0,0,0,35,0,44,3,7,2,0,1,0,3.0,6.0,3,6,0,1,1,2,2,1,14,4,100,0,3,24,2,30.0,0,0,0,0,0,1,0


In [2]:
from backend.playoff_pool.players import get_defense_stats

year = 2025
weekly_stats = nfl.load_player_stats([year]).to_pandas()
weekly_stats = pd.concat([weekly_stats, get_defense_stats(year)])
weekly_stats = weekly_stats[weekly_stats["season_type"] == "POST"]
weekly_stats

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr,gsis_id,full_name,timeouts,points_allowed,points_allowed_0,points_allowed_1_6,points_allowed_7_13,points_allowed_14_20,points_allowed_21_27,points_allowed_28_34,points_allowed_35_plus
18539,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,PIT,HOU,17.0,33.0,146.0,0.0,1.0,4.0,-36.0,2.0,1.0,290.0,73.0,6.0,-26.607500,-7.442237,0.0,0.503448,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,11,0,0,0,0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,1.84,1.84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18540,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,BUF,JAX,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,0.0,0.0,50.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50;47,None,None,97.0,0.0,0.0,3.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18541,00-0026498,M.Stafford,Matthew Stafford,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,LA,CAR,24.0,42.0,304.0,3.0,1.0,1.0,-9.0,0.0,0.0,498.0,111.0,16.0,4.997591,-4.305511,0.0,0.610442,2.0,0.0,0.0,0.0,0.0,1.0,-1.509365,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,22.16,22.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18542,00-0027114,T.Morstead,Thomas Morstead,P,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,SF,PHI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0

# Update draftable players

In [3]:
from asgiref.sync import sync_to_async
from backend.playoff_pool.players import upload_playoff_players_to_db

positions_to_keep = [
    "QB",
    "K",
    "DE",
    "P",
    "OT",
    "OLB",
    "DT",
    "LB",
    "MLB",
    "G",
    "LS",
    "FB",
    "WR",
    "TE",
    "CB",
    "NT",
    "S",
    "SAF",
    "RB",
    "FS",
    "DB",
    "ILB",
    "C",
    "DL",
    "OL",
    "DST",
]

year = 2025
await sync_to_async(upload_playoff_players_to_db)(
    year=year, positions_to_keep=positions_to_keep
)

/Users/pillades/Library/CloudStorage/OneDrive-Paramount/GitHub/espn-api-v3/backend/playoff_pool/players.py:232: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playoff_team_stats["fantasy_points"] = playoff_team_stats.apply(


1356

In [19]:
year = 2025

In [11]:
from backend.playoff_pool.players import get_all_playoff_players

positions_to_keep = [
    "QB",
    "K",
    "DE",
    "P",
    "OT",
    "OLB",
    "DT",
    "LB",
    "MLB",
    "G",
    "LS",
    "FB",
    "WR",
    "TE",
    "CB",
    "NT",
    "S",
    "SAF",
    "RB",
    "FS",
    "DB",
    "ILB",
    "C",
    "DL",
    "OL",
    "DST",
]
playoff_players = get_all_playoff_players(
    year=year, positions_to_keep=positions_to_keep
)
playoff_players.sort_values(by=["draft_value", "fantasy_points"], ascending=False).head(
    20
)

playoff_players[playoff_players["position"] == "DB"]

/Users/pillades/Library/CloudStorage/OneDrive-Paramount/GitHub/espn-api-v3/backend/playoff_pool/players.py:229: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playoff_team_stats["fantasy_points"] = playoff_team_stats.apply(


,gsis_id,player_id,full_name,team,position,fantasy_points,draft_value
369,00-0036349,00-0036349,Kam Curl,LA,DB,113.5,34.5
412,00-0036564,00-0036564,Talanoa Hufanga,DEN,DB,109.5,30.5
556,00-0037253,00-0037253,Marcus Jones,NE,DB,106.0,27.0
870,00-0039343,00-0039343,Kamari Lassiter,HOU,DB,103.5,24.5
1002,00-0039841,00-0039841,Cooper DeJean,PHI,DB,103.5,24.5
...,...,...,...,...,...,...,...
1267,00-0040612,0,Malik Dixon-Williams,LA,DB,0.0,0.0
1291,00-0040659,0,Derrick Canteen,SF,DB,0.0,0.0
1298,00-0040681,0,R.J. Moten,NE,DB,0.0,0.0
1331,00-0040750,0,Kam Alexander,PIT,DB,0.0,0.0


In [ ]:
# Divide fantasy points by the highest value by position
# to get a normalized score between 0 and 1
playoff_players["fantasy_points_adj"] = (
    playoff_players["fantasy_points"]
    / playoff_players.groupby("position")["fantasy_points"].transform("max")
    * 100
)

playoff_players.sort_values(
    by=["fantasy_points_adj", "fantasy_points"], ascending=False
).head(20)

,gsis_id,player_id,full_name,team,position,fantasy_points,fantasy_points_adj
203,00-0034857,00-0034857,Josh Allen,BUF,QB,364.62,100.000000
93,00-0033280,00-0033280,Christian McCaffrey,SF,RB,318.10,100.000000
823,00-0039075,00-0039075,Puka Nacua,LA,WR,246.00,100.000000
39,00-0031492,00-0031492,Jason Myers,SEA,K,192.00,100.000000
12,SEA,SEA,SEA D/ST,SEA,DST,143.00,100.000000
157,00-0034351,00-0034351,Dallas Goedert,PHI,TE,127.10,100.000000
707,00-0038543,00-0038543,Jaxon Smith-Njigba,SEA,WR,243.90,99.146341
77,00-0032726,00-0032726,Ka'imi Fairbairn,HOU,K,189.00,98.437500
5,00-0026498,00-0026498,Matthew Stafford,LA,QB,351.38,96.368822
1005,00-0039851,00-0039851,Drake Maye,NE,QB,350.96,96.253634


In [ ]:
from backend.playoff_pool.players import PLAYOFF_TEAMS

set(PLAYOFF_TEAMS[year]) - set(playoff_players["team"].unique())

set()

In [ ]:
player_stats = nfl.load_player_stats([2025]).to_pandas()
player_stats[player_stats["week"] == 18]

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr
17472,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,18,REG,PIT,BAL,31,47,294,1,0,2,-4,0,0,325,194,14,7.450879,1.217384,0,0.904615,1,20,0,0,0,1,1.890553,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,17.76,17.76
17473,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2025,18,REG,BUF,NYJ,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,3,3,0,0,1.0,0,0,0,0,0,0.00,0.00
17474,00-0026158,J.Flacco,Joe Flacco,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,18,REG,CIN,CLE,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,1,2,0,0,0,1,0.288097,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.20,0.20
17475,00-0026190,C.Campbell,Calais Campbell,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,2025,18,REG,ARI,LA,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,1,0,1,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.00,0.00
17476,00-0026300,J.Johnson,Josh Johnson,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,18,REG,WAS,PHI,14,22,131,1,1,0,0,0,0,123,86,9,4.471365,6.251351,0,1.065041,9,45,1,1,1,5,-3.906335,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,15.74,15.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [ ]:
from backend.playoff_pool.scoring import (
    RELEVANT_SCORING_STATS,
    calculate_fantasy_points,
)


player_stats = nfl.load_player_stats([2024]).to_pandas()
player_stats["fantasy_points"] = player_stats.apply(
    lambda row: calculate_fantasy_points(row, RELEVANT_SCORING_STATS),
    axis=1,
)

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,NYJ,SF,13,21,167,1,1,1,-5,0,0,164,93,8,3.258283,-5.897989,0,1.018293,1,-1,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.00,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,8.58,8.58
1,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,ARI,BUF,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.00,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,2,0,0,31.0,1.0,0,1,1,0,0,0,0,0,0,0,0,0,29;31,None,None,60,0,0,2,2,0,0,1.0,0,0,0,0,0,0.00,0.00
2,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,TEN,CHI,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.00,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,40.0,1.0,0,0,0,1,0,0,0,0,0,0,0,0,40,None,None,40,0,0,2,2,0,0,1.0,0,0,0,0,0,0.00,0.00
3,00-0026190,C.Campbell,Calais Campbell,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,MIA,JAX,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.00,0.000000,0.000000,0,3,0,0,2,14,0,1.0,13.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.00,0.00
4,00-0026498,M.Stafford,Matthew Stafford,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,LA,DET,34,49,317,1,1,2,-13,0,0,290,171,15,4.536421,0.365350,0,1.093103,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.00,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,14.68,14.68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

# Get stats by week

In [2]:
import os

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"


from backend.playoff_pool.models import League

league = League.objects.get(id=20)
league

<League: Vantage 2026 (Created by desidezdez)>

In [3]:
# Get drafted players
from backend.playoff_pool.models import DraftedTeam

drafted_players = DraftedTeam.objects.filter(league=league)
drafted_players

<QuerySet [<DraftedTeam: Christian McCaffrey (RB) -> Jeff Reh in Vantage 2026>, <DraftedTeam: James Cook (RB) -> Joe Lion in Vantage 2026>, <DraftedTeam: Kyren Williams (RB) -> Desi Pilla in Vantage 2026>, <DraftedTeam: Travis Etienne (RB) -> Dan Villeneuve in Vantage 2026>, <DraftedTeam: Puka Nacua (WR) -> Dan Villeneuve in Vantage 2026>, <DraftedTeam: Jaxon Smith-Njigba (WR) -> Ezana Negash in Vantage 2026>, <DraftedTeam: Josh Jacobs (RB) -> Joe Lion in Vantage 2026>, <DraftedTeam: Saquon Barkley (RB) -> Joe Fletcher in Vantage 2026>, <DraftedTeam: D'Andre Swift (RB) -> Ezana Negash in Vantage 2026>, <DraftedTeam: Josh Allen (QB) -> John Quinn in Vantage 2026>, <DraftedTeam: Jaylen Warren (RB) -> Christian Tierney in Vantage 2026>, <DraftedTeam: TreVeyon Henderson (RB) -> Dave Krenn in Vantage 2026>, <DraftedTeam: Matthew Stafford (QB) -> Dave Krenn in Vantage 2026>, <DraftedTeam: Drake Maye (QB) -> Jeff Reh in Vantage 2026>, <DraftedTeam: Kenneth Walker III (RB) -> Dave Krenn in Van

In [3]:
from backend.playoff_pool.scoring import (
    calculate_fantasy_points,
    get_league_scoring_settings,
)

year = 2025

week_map = {
    19: "WC",
    20: "DIV",
    21: "CON",
    22: "SB",
}

# Get weekly stats for playoff weeks
weekly_stats = nfl.load_player_stats([year]).to_pandas()
weekly_stats = weekly_stats[weekly_stats["season_type"] == "POST"]

# Get custom scoring settings
scoring_settings = get_league_scoring_settings(league=league)

# Map weeks to game types based on playoff structure
weekly_stats["fantasy_points"] = weekly_stats.apply(
    lambda row: calculate_fantasy_points(row, scoring_settings),
    axis=1,
)

weekly_stats[["player_id", "player_display_name", "week", "fantasy_points"]].head(20)
weekly_stats[weekly_stats["player_display_name"] == "Drake Maye"]

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr
18871,00-0039851,D.Maye,Drake Maye,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,NE,LAC,17,29,268,1,1,5,-33,2,1,253,136,12,-0.182121,-3.344614,0,1.059289,10,66,0,0,0,4,2.300692,0,0,1,0,0,0,0,12,0,0,-1.643551,0,0.0,0.037037,0.045283,0.087254,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,18.32,17.32


# Get defense stats by week

In [7]:
from backend.playoff_pool.players import PLAYOFF_TEAMS

year = 2025

team_stats = nfl.load_team_stats(seasons=[year]).to_pandas()

playoff_team_stats = team_stats[
    team_stats["team"].isin(PLAYOFF_TEAMS[year]) & (team_stats["season_type"] != "REG")
]
playoff_team_stats

schedules = nfl.load_schedules(seasons=[year]).to_pandas()
schedules

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2025_01_DAL_PHI,2025,REG,1,2025-09-04,Thursday,20:20,DAL,20.0,PHI,24.0,Home,4.0,44.0,0.0,2025090400,59843.0,None,202509040phi,28418.0,401772510,6734.0,7,7,330,-425,8.5,-110,-110,47.5,-110,-110,1,outdoors,grass,75.0,11.0,00-0033077,00-0036389,Dak Prescott,Jalen Hurts,Brian Schottenheimer,Nick Sirianni,Shawn Smith,PHI00,Lincoln Financial Field
1,2025_01_KC_LAC,2025,REG,1,2025-09-05,Friday,20:00,KC,21.0,LAC,27.0,Neutral,6.0,48.0,0.0,2025090500,59844.0,None,202509050sdg,28419.0,401772714,6735.0,7,7,-175,145,-3.0,-120,100,47.5,-110,-110,1,dome,,NaN,NaN,00-0033873,00-0036355,Patrick Mahomes,Justin Herbert,Andy Reid,Jim Harbaugh,Carl Cheffers,LAX01,SoFi Stadium
2,2025_01_TB_ATL,2025,REG,1,2025-09-07,Sunday,13:00,TB,23.0,ATL,20.0,Home,-3.0,43.0,0.0,2025090700,59845.0,None,202509070atl,28420.0,401772830,6736.0,7,7,-115,-105,-1.5,-102,-118,47.5,-105,-115,1,closed,fieldturf,NaN,NaN,00-0034855,00-0039917,Baker Mayfield,Michael Penix,Todd Bowles,Raheem Morris,Land Clark,ATL97,Mercedes-Benz Stadium
3,2025_01_CIN_CLE,2025,REG,1,2025-09-07,Sunday,13:00,CIN,17.0,CLE,16.0,Home,-1.0,33.0,0.0,2025090701,59846.0,None,202509070cle,28421.0,401772829,6737.0,7,7,-238,195,-5.5,-105,-115,47.5,100,-120,1,outdoors,grass,63.0,10.0,00-0036442,00-0026158,Joe Burrow,Joe Flacco,Zac Taylor,Kevin Stefanski,Adrian Hill,CLE00,FirstEnergy Stadium
4,2025_01_MIA_IND,2025,REG,1,2025-09-07,Sunday,13:00,MIA,8.0,IND,33.0,Home,25.0,41.0,0.0,2025090702,59847.0,None,202509070clt,28422.0,401772719,6738.0,7,7,-108,-112,1.5,-118,-102,47.5,-120,100,0,closed,fieldturf,NaN,NaN,00-0036212,00-0035710,Tua Tagovailoa,Daniel Jones,Mike McDaniel,Shane Steichen,Brad Allen,IND00,Lucas Oil Stadium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,2025_19_LA_CAR,2025,WC,19,2026-01-10,Saturday,16:30,LA,34.0,CAR,31.0,Home,-3.0,65.0,0.0,2026011000,60164.0,None,202601100car,NaN,401772979,7006.0,6,7,-550,410,-10.0,-108,-112,44.5,-108,-112,0,outdoors,grass,72.0,16.0,00-0026498,00-0039150,Matthew Stafford,Bryce Young,Sean McVay,Dave Canales,Clete Blakeman,CAR00,Bank of America Stadium
278,2025_20_BUF_DEN,2025,DIV,20,2026-01-17,Saturday,16:30,BUF,NaN,DEN,NaN,Home,NaN,NaN,NaN,2026011700,NaN,None,202601170den,NaN,401772982,NaN,6,13,-108,-112,1.5,-118,-102,46.5,-105,-115,0,outdoors,grass,NaN,NaN,00-0034857,00-0039732,Josh Allen,Bo Nix,Sean McDermott,Sean Payton,None,DEN00,Empower Field at Mile High
279,2025_20_SF_SEA,2025,DIV,20,2026-01-17,Saturday,20:00,SF,NaN,SEA,NaN,Home,NaN,NaN,NaN,2026011701,NaN,None,202601170sea,NaN,401772984,NaN,6,14,280,-355,7.5,-112,-108,45.5,-112,-108,1,outdoors,fieldturf,NaN,NaN,00-0037834,00-0034869,Brock Purdy,Sam Darnold,Kyle Shanahan,Mike Macdonald,None,SEA00,Lumen Field
280,2025_20_LA_CHI,2025,DIV,20,2026-01-18,Sunday,18:30,LA,NaN,CHI,NaN,Home,NaN,NaN,NaN,2026011801,NaN,None,202601180chi,NaN,401772985,NaN,8,8,-198,164,-3.5,-115,-105,50.5,-105,-115,0,outdoors,grass,NaN,NaN,00-0026498,00-0039918,Matthew Stafford,Caleb Williams,Sean McVay,Ben Johnson,None,CHI98,Soldier Field


In [7]:
import os

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"


from backend.playoff_pool.models import League

league = League.objects.get(id=17)
league

<League: defense only (Created by testuser)>

In [34]:
import importlib
import backend.playoff_pool.scoring
import backend.playoff_pool.nfl_utils

importlib.reload(backend.playoff_pool.scoring)
importlib.reload(backend.playoff_pool.nfl_utils)

from backend.playoff_pool.nfl_utils import calculate_player_playoff_points

calculate_player_playoff_points(league)

Adding stat special_teams_tds with value 0.0
Adding stat def_tds with value 0.0
Adding stat fumble_recovery_tds with value 0.0
Adding stat def_tackles_solo with value 0.0
Adding stat def_tackles_with_assist with value 0.0
Adding stat def_tackle_assists with value 0.0
Adding stat def_tackles_for_loss with value 0.0
Adding stat def_tackles_for_loss_yards with value 0.0
Adding stat def_fumbles_forced with value 0.0
Adding stat def_fumbles with value 0.0
Adding stat def_interceptions with value 6.0
Adding stat def_interception_yards with value 0.0
Adding stat def_sacks with value 2.0
Adding stat def_sack_yards with value 0.0
Adding stat def_qb_hits with value 0.0
Adding stat def_pass_defended with value 0.0
Adding stat def_safeties with value 0.0
Adding stat fumble_recovery_own with value 0.0
Adding stat fumble_recovery_yards_own with value 0.0
Adding stat fumble_recovery_opp with value 2.0
Adding stat fumble_recovery_yards_opp with value 0.0
Adding stat punt_returns with value 0.0
Adding 

{'DEN': {'player_info': {'gsis_id': 'DEN',
   'player_name': 'DEN D/ST',
   'position': 'DST',
   'nfl_team': 'DEN',
   'team_name': "testuser's Team",
   'user': 'testuser'},
  'round_points': {'WC': 2.0, 'DIV': 0.0, 'CON': 0.0, 'SB': 0.0},
  'total_points': 2.0},
 'PHI': {'player_info': {'gsis_id': 'PHI',
   'player_name': 'PHI D/ST',
   'position': 'DST',
   'nfl_team': 'PHI',
   'team_name': 'desi',
   'user': None},
  'round_points': {'WC': 14.0, 'DIV': 111.0, 'CON': 111.0, 'SB': 118.0},
  'total_points': 354.0},
 'HOU': {'player_info': {'gsis_id': 'HOU',
   'player_name': 'HOU D/ST',
   'position': 'DST',
   'nfl_team': 'HOU',
   'team_name': "testuser's Team",
   'user': 'testuser'},
  'round_points': {'WC': 22.0, 'DIV': 105.0, 'CON': 0.0, 'SB': 0.0},
  'total_points': 127.0},
 'BAL': {'player_info': {'gsis_id': 'BAL',
   'player_name': 'BAL D/ST',
   'position': 'DST',
   'nfl_team': 'BAL',
   'team_name': 'desi',
   'user': None},
  'round_points': {'WC': 13.0, 'DIV': 101.0, '

# D/ST 4th down stops

In [2]:
from yahoo_oauth import OAuth2

import yahoo_fantasy_api as yfa

yahoo_client_id = "dj0yJmk9N0NNRDlIeUhlU0VjJmQ9WVdrOWIxQktaSGxvV2tvbWNHbzlNQT09JnM9Y29uc3VtZXJzZWNyZXQmc3Y9MCZ4PTI2"
yahoo_client_secret = "66c6a02d2f7a9e33d7b6c4f0e411118f67a1bdc6"

yahoo_client = OAuth2(yahoo_client_id, yahoo_client_secret, from_file="secrets.json")
yahoo_client

[2026-01-20 10:23:22,968 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-01-20 10:23:22,969 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 853.5108580589294
[2026-01-20 10:23:22,969 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


In [3]:
gm = yfa.Game(yahoo_client, "nfl")
gm.league_ids()

['461.l.1515417', '461.l.1515422']

In [4]:
league = yfa.League(yahoo_client, "461.l.1515417")

def_dict = {}
for status in ["A", "T", "K"]:
    defenses = league._fetch_players(status=status, position="DEF")
    for defense in defenses:
        def_dict[defense["name"]] = defense["player_id"]

def_dict

{'Falcons': 100001,
 'Bears': 100003,
 'Bengals': 100004,
 'Cowboys': 100006,
 'Broncos': 100007,
 'Packers': 100009,
 'Titans': 100010,
 'Colts': 100011,
 'Raiders': 100013,
 'Rams': 100014,
 'Dolphins': 100015,
 'Patriots': 100017,
 'Saints': 100018,
 'Giants': 100019,
 'Jets': 100020,
 'Cardinals': 100022,
 '49ers': 100025,
 'Buccaneers': 100027,
 'Commanders': 100028,
 'Panthers': 100029,
 'Jaguars': 100030,
 'Bills': 100002,
 'Browns': 100005,
 'Lions': 100008,
 'Chiefs': 100012,
 'Vikings': 100016,
 'Eagles': 100021,
 'Steelers': 100023,
 'Chargers': 100024,
 'Seahawks': 100026,
 'Ravens': 100033,
 'Texans': 100034}

In [13]:
YAHOO_DEFENSE_ID_MAP = {
    "ATL": 100001,
    "CHI": 100003,
    "CIN": 100004,
    "DAL": 100006,
    "DEN": 100007,
    "GB": 100009,
    "TEN": 100010,
    "IND": 100011,
    "LV": 100013,
    "LA": 100014,
    "MIA": 100015,
    "NE": 100017,
    "NO": 100018,
    "NYG": 100019,
    "NYJ": 100020,
    "ARI": 100022,
    "SF": 100025,
    "TB": 100027,
    "WAS": 100028,
    "CAR": 100029,
    "JAX": 100030,
    "BUF": 100002,
    "CLE": 100005,
    "DET": 100008,
    "KC": 100012,
    "MIN": 100016,
    "PHI": 100021,
    "PIT": 100023,
    "LAC": 100024,
    "SEA": 100026,
    "BAL": 100033,
    "HOU": 100034,
}

In [ ]:
# league.player_stats(player_ids=list(YAHOO_DEFENSE_ID_MAP.values())[:1], req_type="week", week=[19])
# league.player_stats(player_ids=[100034], req_type="week", week=[10])

# league._fetch_plyr_stats([100034], "week", week=19, date=None, season=2025)
# Instead of week-based query

for player_name, player_id in YAHOO_DEFENSE_ID_MAP.items():
    stats = league._fetch_plyr_stats(
        [player_id], req_type="date", date="2025-01-11", week=None, season=None
    )
    print(player_name, stats)

ATL [{'player_id': 100001, 'name': 'Falcons', 'position_type': 'DT'}]
CHI [{'player_id': 100003, 'name': 'Bears', 'position_type': 'DT'}]
CIN [{'player_id': 100004, 'name': 'Bengals', 'position_type': 'DT'}]
DAL [{'player_id': 100006, 'name': 'Cowboys', 'position_type': 'DT'}]
DEN [{'player_id': 100007, 'name': 'Broncos', 'position_type': 'DT'}]
GB [{'player_id': 100009, 'name': 'Packers', 'position_type': 'DT'}]
TEN [{'player_id': 100010, 'name': 'Titans', 'position_type': 'DT'}]
IND [{'player_id': 100011, 'name': 'Colts', 'position_type': 'DT'}]
LV [{'player_id': 100013, 'name': 'Raiders', 'position_type': 'DT'}]
LA [{'player_id': 100014, 'name': 'Rams', 'position_type': 'DT'}]
MIA [{'player_id': 100015, 'name': 'Dolphins', 'position_type': 'DT'}]
NE [{'player_id': 100017, 'name': 'Patriots', 'position_type': 'DT'}]
NO [{'player_id': 100018, 'name': 'Saints', 'position_type': 'DT'}]
NYG [{'player_id': 100019, 'name': 'Giants', 'position_type': 'DT'}]
NYJ [{'player_id': 100020, 'name'

In [5]:
from backend.playoff_pool.players import get_defense_stats


year = 2025
defense_stats = get_defense_stats(year)
defense_stats

,season,week,gsis_id,full_name,position,player_id,team,season_type,special_teams_tds,def_tds,fumble_recovery_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_fumbles,def_interceptions,def_interception_yards,def_sacks,def_sack_yards,def_qb_hits,def_pass_defended,def_safeties,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,misc_yards,penalties,penalty_yards,timeouts,team_score,points_allowed,points_allowed_0,points_allowed_1_6,points_allowed_7_13,points_allowed_14_20,points_allowed_21_27,points_allowed_28_34,points_allowed_35_plus
0,2025,1,ARI,ARI D/ST,DST,ARI,ARI,REG,0,0,0,36,2,36,3,10,0,0,0,0,1.0,6.0,2,10,0,0,0,0,0,3,30,3,73,0,9,54,0,20.0,13.0,0,0,1,0,0,0,0
1,2025,1,ATL,ATL D/ST,DST,ATL,ATL,REG,0,0,0,27,1,25,2,3,0,0,0,0,1.0,8.0,3,7,0,1,2,0,0,2,29,5,125,0,8,55,6,20.0,23.0,0,0,0,0,1,0,0
2,2025,1,BAL,BAL D/ST,DST,BAL,BAL,REG,0,0,0,51,2,22,3,8,1,0,0,0,1.0,5.0,3,4,0,1,1,0,0,1,6,6,167,0,7,51,5,40.0,41.0,0,0,0,0,0,0,1
3,2025,1,BUF,BUF D/ST,DST,BUF,BUF,REG,0,0,0,44,0,8,5,45,2,0,0,0,2.0,15.0,4,1,0,0,0,1,1,1,0,7,193,0,5,38,6,41.0,40.0,0,0,0,0,0,0,1
4,2025,1,CAR,CAR D/ST,DST,CAR,CAR,REG,0,0,0,42,0,20,1,2,0,0,1,10,0.0,0.0,1,4,0,0,0,0,0,2,0,2,43,0,4,35,6,10.0,26.0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559,2025,20,HOU,HOU D/ST,DST,HOU,HOU,POST,0,0,0,40,1,35,9,48,4,0,1,0,5.0,36.0,2,6,0,0,0,2,2,3,29,4,88,0,4,32,3,16.0,28.0,0,0,0,0,0,1,0
560,2025,20,LA,LA D/ST,DST,LA,LA,POST,0,0,0,41,1,47,4,8,0,0,3,14,0.0,0.0,4,9,0,2,2,0,0,0,0,4,88,0,1,5,6,20.0,17.0,0,0,0,1,0,0,0
561,2025,20,NE,NE D/ST,DST,NE,NE,POST,0,1,0,37,1,29,6,25,1,0,4,26,3.0,19.0,9,14,0,2,2,1,1,4,53,4,89,0,3,48,4,28.0,16.0,0,0,0,1,0,0,0
562,2025,20,SEA,SEA D/ST,DST,SEA,SEA,POST,1,0,0,38,0,28,4,21,3,0,1,0,2.0,16.0,5,7,0,0,0,2,2,0,0,2,126,0,2,5,5,41.0,6.0,0,1,0,0,0,0,0


In [ ]:
league.player_stats(player_ids=[], req_type="week", week=19)

# `is_eliminated` flag

In [7]:
import importlib
import backend.playoff_pool.scoring
import backend.playoff_pool.players

importlib.reload(backend.playoff_pool.scoring)
importlib.reload(backend.playoff_pool.players)

from backend.playoff_pool.scoring import (
    calculate_fantasy_points,
    get_league_scoring_settings,
)
from backend.playoff_pool.players import get_schedule

year = 2025


# Load the schedule
schedule = get_schedule(year)
schedule["is_eliminated"] = schedule["team_score"] < schedule["opponent_score"]
schedule.tail(10)

,season,week,team,team_score,opponent_score,is_eliminated
554,2025,19,GB,27.0,31.0,True
555,2025,19,BUF,27.0,24.0,False
556,2025,19,SF,23.0,19.0,False
557,2025,19,LAC,3.0,16.0,True
558,2025,19,HOU,30.0,6.0,False
559,2025,19,LA,34.0,31.0,False
560,2025,20,BUF,NaN,NaN,False
561,2025,20,SF,NaN,NaN,False
562,2025,20,LA,NaN,NaN,False
563,2025,20,HOU,NaN,NaN,False


In [9]:
# Get weekly stats for playoff weeks
weekly_stats = nfl.load_player_stats([year]).to_pandas()
weekly_stats = weekly_stats[weekly_stats["season_type"] == "POST"]
weekly_stats.tail()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr
18932,00-0040735,L.Burden,Luther Burden III,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,CHI,GB,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,1,-4,0,0,0,0,-1.233243,0,3,7,42,0,0,0,86,21,3,1.001998,0,0.488372,0.152174,0.154122,0.336146,0,1,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,3.8,6.8
18933,00-0040740,O.Trapilo,Ozzy Trapilo,OT,OL,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,CHI,GB,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0
18934,00-0040745,A.Ersery,Aireontae Ersery,OT,OL,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,HOU,PIT,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,5,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0
18935,00-0040746,N.Scourton,Nic Scourton,LB,LB,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,CAR,LA,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,1,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0
18936,None,None,None,None,None,None,2025,19,POST,CAR,LA,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,25,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0


In [11]:
weekly_stats = weekly_stats.merge(
    schedule[["season", "week", "team", "team_score", "opponent_score"]],
    on=["season", "week", "team"],
    how="left",
)
weekly_stats["is_eliminated"] = (
    weekly_stats["team_score"] < weekly_stats["opponent_score"]
)
weekly_stats.tail()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr,is_eliminated,team_score,opponent_score
393,00-0040735,L.Burden,Luther Burden III,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,CHI,GB,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,1,-4,0,0,0,0,-1.233243,0,3,7,42,0,0,0,86,21,3,1.001998,0,0.488372,0.152174,0.154122,0.336146,0,1,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,3.8,6.8,False,31.0,27.0
394,00-0040740,O.Trapilo,Ozzy Trapilo,OT,OL,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,CHI,GB,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0,False,31.0,27.0
395,00-0040745,A.Ersery,Aireontae Ersery,OT,OL,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,HOU,PIT,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,5,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0,False,30.0,6.0
396,00-0040746,N.Scourton,Nic Scourton,LB,LB,https://static.www.nfl.com/image/upload/f_auto...,2025,19,POST,CAR,LA,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,1,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0,True,31.0,34.0
397,None,None,None,None,None,None,2025,19,POST,CAR,LA,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,25,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0.0,0.0,True,31.0,34.0
